# Postprocess into rasters and vector datasets




In [13]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np
import xdem
from osgeo import gdal, ogr, osr
import rasterstats
from osgeo_utils import gdal_calc


import subkart

In [14]:
nodata = 255
crs = "EPSG:25833"

In [15]:
classifier = subkart.utils.load_classifier()

## Post processing

In [16]:
predict_file_unmapped = "predict_unmapped.tif"
prob_file = "3band_probability.tif"

predict_file_remapped = "predict_remapped.tif"


In [17]:
fname = subkart.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}", "norge", "latest", crs.split(":")[1]
)
predict_file = f"{fname}.tif"

fname_prob = subkart.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}-probability", "norge", "latest", crs.split(":")[1]
)
prob_file_processed = f"{fname_prob}.tif"

In [18]:
# predict_unmapped.tif and 3band_probability.tif are produced by 03_predict.ipynb
gdal.UseExceptions()
assert Path(predict_file_unmapped).exists(), f"{predict_file_unmapped} not found – run 03_predict.ipynb first"
assert Path(prob_file).exists(), f"{prob_file} not found – run 03_predict.ipynb first"
print(f"Using {predict_file_unmapped} and {prob_file}")

# Create processed prediction raster:

Remap class 1 (blanding) → highest-probability class (0=løsbunn or 2=fastbunn) using gdal_calc

In [7]:
subkart.utils.remap_prediction(predict_file_unmapped, prob_file, predict_file_remapped, nodata=nodata)

## Create processed 1-band probability raster from the 3-band source

* class 0 (løsbunn)        band 1 = P(class=0)
* class 2 (fastbunn)       band 3 = P(class=2)

In [8]:
subkart.utils.create_probability_raster(predict_file_remapped, prob_file, prob_file_processed, nodata=nodata)

## Filter isolated low-probability pixels

Use `gdal.SieveFilter` (threshold=1, 4-connected) to replace single isolated pixels with the
surrounding class, then update `prob_file_processed` for changed pixels.

In [9]:
PROB_THRESHOLD = 0.60  # Filter probability threshold for filtering noise pixels

In [10]:
subkart.utils.filter_isolated_pixels(
    predict_file_remapped, prob_file_processed, predict_file, nodata=nodata, prob_threshold=PROB_THRESHOLD
)

## Vectorize processed prediction raster

In [11]:
subkart.vectorize.with_gdal(
    predict_file, "polygons_processed.gpkg", epsg_code=int(crs.split(":")[1])
)

gdf = gpd.read_file("polygons_processed.gpkg").explode()

reverse_map = {v: k for k, v in subkart.labelling.BUNNTYPE_MAPPING.items()}
gdf["BunnType"] = gdf["DN"].map(reverse_map)

# Compute mean probability per polygon from the processed 1-band probability raster
stats = rasterstats.zonal_stats(
    gdf,
    prob_file_processed,
    stats=["mean"],
    nodata=-9999,
)
gdf["Sannsynlighet"] = [s["mean"]*100 for s in stats]

gdf.to_file(f"{fname}.gpkg", driver="GPKG", layer="bunntyper")
gdf.to_parquet(f"{fname}.geo.parquet", compression="snappy")


Polygons saved to polygons_processed.gpkg


In [12]:
subkart.utils.to_postgis(gdf, fname)

Table nisjedata_substrat_xgbclassifier_norge_latest uploaded to PostGIS.


## Final dataproduct: merge authoritative classifications

Replace model predictions with authoritative polygons from the *klassifisering* dataset.
`blanding` (DN=1) is mapped to `løsbunn` (DN=0) since the model has no blanding class.
Probability is set to 100 for all authoritative polygons.

In [21]:
import pandas as pd
import numpy as np
import shapely
from shapely.strtree import STRtree
from collections import defaultdict

# LM_DK habitat codes per BunnType (applied before blanding is remapped)
LM_DK_MAP = {"løsbunn": "DK_AB, DK_C, DK_D", "fastbunn": "DK_EFGY", "blanding": "DK_0"}

# Load authoritative classifications – only needed columns
klass = gpd.read_parquet(
    "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat-klassifisering_norge_latest_25833.geo.parquet",
    columns=["BunnType", "geometry"],
)

# Assign LM_DK before remapping blanding so blanding keeps DK_0
klass["LM_DK"] = klass["BunnType"].map(LM_DK_MAP)

# Map blanding (not in model) → løsbunn
klass["BunnType"] = klass["BunnType"].replace({"blanding": "løsbunn"})
klass["DN"] = klass["BunnType"].map({"løsbunn": 0, "fastbunn": 2})
klass["Sannsynlighet"] = 100.0
klass["Kilde"] = "NGU - Bunnsedimenter (kornstørrelse), detaljert"

# Load model predictions – only needed columns
pred = gpd.read_parquet(
    f"gs://niva-geodata/MarintNaturKart/results/{fname}.geo.parquet",
    columns=["DN", "BunnType", "Sannsynlighet", "geometry"],
)
pred["LM_DK"] = pred["BunnType"].map(LM_DK_MAP)
pred["Kilde"] = "NIVA - Substrat Modell"



In [22]:
# --- Fast erase: bulk spatial join finds all intersecting pairs in one GEOS call ---
klass_geoms = klass.geometry.values
pred_geoms = pred.geometry.values.copy()

tree = STRtree(klass_geoms)
pred_hit_idxs, klass_hit_idxs = tree.query(pred_geoms, predicate="intersects")

# Group klass indices by pred polygon for efficient per-polygon difference
pred_to_klass = defaultdict(list)
for p, k in zip(pred_hit_idxs.tolist(), klass_hit_idxs.tolist()):
    pred_to_klass[p].append(k)

n_clip = len(pred_to_klass)
print(f"Clipping {n_clip:,} of {len(pred_geoms):,} prediction polygons...")
for i, (pred_i, klass_is) in enumerate(pred_to_klass.items(), 1):
    if i % 1000 == 0:
        print(f"  {i:,} / {n_clip:,}")
    eraser = shapely.unary_union(klass_geoms[klass_is])
    pred_geoms[pred_i] = shapely.difference(pred_geoms[pred_i], eraser)

pred = pred.set_geometry(pred_geoms)
pred_remaining = pred[~pred.geometry.is_empty].explode(index_parts=False)

COLS = ["DN", "BunnType", "LM_DK", "Sannsynlighet", "Kilde", "geometry"]
final = pd.concat(
    [pred_remaining[COLS], klass[COLS]],
    ignore_index=True,
)

fname_final = subkart.utils.to_filename("nisjedata-substrat", "norge", "2026", crs.split(":")[1])
final.to_parquet(f"{fname_final}.geo.parquet", compression="snappy")
final.to_file(f"{fname_final}.gpkg", driver="GPKG", layer="bunntyper")

Exception ignored in: <function tqdm.__del__ at 0x7fdba943ccc0>
Traceback (most recent call last):
  File "/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/tqdm/notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


Clipping 43,673 of 396,429 prediction polygons...
  1,000 / 43,673
  2,000 / 43,673
  3,000 / 43,673
  4,000 / 43,673
  5,000 / 43,673
  6,000 / 43,673
  7,000 / 43,673
  8,000 / 43,673
  9,000 / 43,673
  10,000 / 43,673
  11,000 / 43,673
  12,000 / 43,673
  13,000 / 43,673
  14,000 / 43,673
  15,000 / 43,673
  16,000 / 43,673
  17,000 / 43,673
  18,000 / 43,673
  19,000 / 43,673
  20,000 / 43,673
  21,000 / 43,673
  22,000 / 43,673
  23,000 / 43,673
  24,000 / 43,673
  25,000 / 43,673
  26,000 / 43,673
  27,000 / 43,673
  28,000 / 43,673
  29,000 / 43,673
  30,000 / 43,673
  31,000 / 43,673
  32,000 / 43,673
  33,000 / 43,673
  34,000 / 43,673
  35,000 / 43,673
  36,000 / 43,673
  37,000 / 43,673
  38,000 / 43,673
  39,000 / 43,673
  40,000 / 43,673
  41,000 / 43,673
  42,000 / 43,673
  43,000 / 43,673


In [ ]:
subkart.utils.to_postgis(final, fname_final)